# CS4207 Coding Challenge - NYC Yellow Taxi Dataset

### Group 19

Igor Kochanski 23358459, Tudor Ventuneac 23390131,Emma O'Reilly 23304472, Sean Fitzgerald 22370595, Hersh Singh 23371137

- [Exploratory Data Analysis](#exploratory-data-analysis)
- [Data Preprocessing](#data-preprocessing)

# Requirements
Requires Python 3.10+

Create virtual environment and activate it

```sh
# Linux/MacOS
python3 -m venv .venv
source .venv/bin/activate

# Windows
python -m venv .venv
.venv\Scripts\activate
```

Install requirements (run cell below to generate requirements.txt)

```
pip install -r requirements.txt
```

In [ ]:
%%writefile requirements.txt
nbstripout
jupyter
ipykernel
ipywidgets
matplotlib
tqdm
requests
scikit-learn
pandas
pyarrow

# Exploratory Data Analysis

Covers every month used by the pipeline (2024-01 to 2026-09, ~123M trips). 

**Train** = 2024-01 to 2025-12, **test** = 2026-01 to 2026-09

In [ ]:
import os
import sys
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from tqdm.auto import tqdm

sys.path.append("../src")
from preprocess import (COLUMNS, PICKUP, DROPOFF, RATE_CODES, TEST_START, TEST_END,
                        MIN_DURATION, MAX_DURATION, MIN_DISTANCE, MAX_DISTANCE, MIN_FARE, MAX_FARE,
                        clean, extract_features, month_of)

FIG_DIR = "../results/eda"
os.makedirs(FIG_DIR, exist_ok=True)

# same months as preprocess.py
files = [f for f in sorted(glob.glob("../data/yellow_tripdata_*.parquet")) if month_of(f) <= TEST_END]

# refine plots
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.titleweight": "bold"})
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
YEAR_COLOURS = {"2024": "tab:blue", "2025": "tab:green", "2026": "tab:orange"}
RATE_NAMES = {0: "missing", 1: "standard", 2: "JFK", 3: "Newark", 4: "Nassau/Westchester",
              5: "negotiated", 6: "group", 99: "unknown"}

## Monthly summary

`month_stats()` summaries monthly data.\
Results are cached in `results/eda/monthly_stats.pkl`.

In [ ]:
BIN = 0.5                                      
EDGES = np.arange(0, MAX_DURATION + BIN, BIN)    
NB = len(EDGES) - 1                              
# columns for the correlation matrix
NUMERIC = ["trip_distance", "fare_amount", "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
           "month_sin", "month_cos", "is_weekend", "duration_min"]


def sums(index, size, y):
    return np.stack([np.bincount(index, minlength=size),
                     np.bincount(index, y, minlength=size),
                     np.bincount(index, y * y, minlength=size)])


def month_stats(f):
    month = month_of(f)

    table = pq.read_table(f)
    nulls = {c: table.column(c).null_count for c in table.column_names} # nulls per column
    n_raw = table.num_rows # row count before cleaning
    raw = table.select(COLUMNS).to_pandas()
    del table

    # row dropping criteria
    duration = (raw[DROPOFF] - raw[PICKUP]).dt.total_seconds() / 60
    start = pd.Timestamp(f"{month}-01")
    problems = {
        "pickup outside month": ~raw[PICKUP].between(start, start + pd.DateOffset(months=1), inclusive="left"),
        "duration < 1 min": duration < MIN_DURATION,
        "duration > 3 hours": duration > MAX_DURATION,
        "distance < 0.1 mi": raw.trip_distance < MIN_DISTANCE,
        "distance > 60 mi": raw.trip_distance > MAX_DISTANCE,
        "fare < $3": raw.fare_amount < MIN_FARE,
        "fare > $250": raw.fare_amount > MAX_FARE,
        "RatecodeID not a known code": ~raw.RatecodeID.fillna(0).isin(RATE_CODES), # missing counts as code 0
    }
    problems = {name: int(rows.sum()) for name, rows in problems.items()}
    extremes = {"max distance": raw.trip_distance.max(), "min fare": raw.fare_amount.min(),
                "max fare": raw.fare_amount.max(), "min duration": duration.min(), "max duration": duration.max()}

    #cleaned trips
    d = clean(raw, f)
    del raw, duration
    y = d.duration_min.to_numpy()

    # used to index the histograms and sums
    hour = d[PICKUP].dt.hour.to_numpy() 
    day = d[PICKUP].dt.day_of_week.to_numpy()                                
    rate = np.searchsorted(RATE_CODES, d.RatecodeID.fillna(0).astype(int).to_numpy())  # position in RATE_CODES
    pu, do = d.PULocationID.to_numpy(), d.DOLocationID.to_numpy() # taxi zone IDs 
    b = np.minimum((y // BIN).astype(int), NB - 1) # duration bin 

    # numeric features
    feats = extract_features(d)
    numeric = np.column_stack([d.trip_distance, d.fare_amount] +
                              [feats[c].astype(float) for c in NUMERIC[2:-1]] + [y])

    return {
        "month": month,
        "columns": list(nulls),
        "nulls": nulls,
        "rows_raw": n_raw,
        "rows_clean": len(d),
        "problems": problems,
        "extremes": extremes,
        "hist_day_hour": np.bincount((day * 24 + hour) * NB + b, minlength=7 * 24 * NB).reshape(7, 24, NB),  
        "hist_rate": np.bincount(rate * NB + b, minlength=len(RATE_CODES) * NB).reshape(-1, NB),         
        "by_hour": sums(hour, 24, y),
        "by_rate": sums(rate, len(RATE_CODES), y),
        "by_pickup": sums(pu, 266, y),
        "by_route": sums(pu * 266 + do, 266 * 266, y),
        "corr": (len(numeric), numeric.sum(0), numeric.T @ numeric),
        # small sample of trips for the scatter plot
        "sample": d[["trip_distance", "RatecodeID", "duration_min"]].sample(min(3000, len(d)), random_state=0),
    }

In [ ]:
# load monthly summaries
CACHE = f"{FIG_DIR}/monthly_stats.pkl"
months = [month_of(f) for f in files]

# use cache or rebuild
stats = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else []
if [s["month"] for s in stats] != months:  # missing or out of date
    stats = [month_stats(f) for f in tqdm(files, unit=" month")]
    pickle.dump(stats, open(CACHE, "wb"))

# boolean masks over months
is_test = np.array([m >= TEST_START for m in months])
SPLITS = {"all": np.ones(len(stats), bool), "train": ~is_test, "test": is_test}


# for combining months 
def total(key, split="all"):
    return sum(s[key] for s, keep in zip(stats, SPLITS[split]) if keep)


def hist_quantile(h, q=0.5):
    c = np.cumsum(h)
    target = q * c[-1]                
    i = np.searchsorted(c, target)     
    below = c[i - 1] if i else 0  
    return EDGES[i] + BIN * (target - below) / h[i]


def r2(groups, split="train"):
    n, s, ss = total(groups, split)
    within = (ss - np.divide(s ** 2, n, out=np.zeros_like(s), where=n > 0)).sum()
    return 1 - within / (ss.sum() - s.sum() ** 2 / n.sum())


# one row per month, for the plots
monthly = pd.DataFrame({
    "rows_raw": [s["rows_raw"] for s in stats],
    "rows_clean": [s["rows_clean"] for s in stats],
    "mean_duration": [s["by_hour"][1].sum() / s["by_hour"][0].sum() for s in stats],  # total minutes / trips
    "median_duration": [hist_quantile(s["hist_rate"].sum(0)) for s in stats],
}, index=months)
monthly["kept"] = monthly.rows_clean / monthly.rows_raw  # share of raw trips that survive cleaning
print(f"{len(stats)} months: {len(stats) - is_test.sum()} train, {is_test.sum()} test")

## 1. Raw data analysis

In [ ]:
print(f"{monthly.rows_raw.sum():,} trips")
for split in ("train", "test"):
    print(f"  {split}: {monthly.rows_raw[SPLITS[split]].sum():,}")

monthly.rows_raw.div(1e6).plot.bar(figsize=(12, 3), color=[YEAR_COLOURS[m[:4]] for m in months],
                                   title="Trips per month (millions): blue 2024, green 2025, orange 2026 (test)")
plt.savefig(f"{FIG_DIR}/01_monthly_volume.png", bbox_inches="tight")

In [ ]:
has = pd.DataFrame({s["month"]: dict.fromkeys(s["columns"], True) for s in stats}).T.fillna(False)
for col in has.columns[~has.all()]:
    print(f"{col!r} only in {has.index[has[col]][0]} to {has.index[has[col]][-1]}")
print("columns the pipeline uses present in every month:", has[COLUMNS].all().all())

In [ ]:
# % missing per column, by year
nulls = pd.DataFrame([s["nulls"] for s in stats], index=months).fillna(0)
by_year = nulls.groupby(nulls.index.str[:4]).sum().div(monthly.rows_raw.groupby(monthly.index.str[:4]).sum(), axis=0) * 100
by_year.loc[:, by_year.sum() > 0].round(2)

In [ ]:
nulls.RatecodeID.div(monthly.rows_raw).mul(100).plot.bar(
    figsize=(12, 3), color=[YEAR_COLOURS[m[:4]] for m in months], title="% of trips missing RatecodeID")
plt.savefig(f"{FIG_DIR}/01_missing_ratecode.png", bbox_inches="tight")

In [ ]:
# the most extreme raw values, by year
extremes = pd.DataFrame([s["extremes"] for s in stats], index=months)
extremes.groupby(extremes.index.str[:4]).agg({"max distance": "max", "min fare": "min", "max fare": "max",
                                              "min duration": "min", "max duration": "max"}).round(1)

**Notes:** 
- 123.5M trips over 33 months (89.9M train, 33.6M test). 
- Files aren't all the same shape: `cbd_congestion_fee` appears from 2025-01 and `request_source` from 2026-06.
- Every column the pipeline reads is in every month.

**Missing values**. 
- `RatecodeID`, `passenger_count` and other sparse columns are missing on the same rows
-  Share increases from ~10% in 2024 to ~27% in 2026.
- Broken values turn up every year: 400,000-mile trips, fares up to \$863k, and durations of -300,000 minutes.

## 2. Bad rows

In [ ]:
# % of raw rows hitting each rule
problems = pd.DataFrame([s["problems"] for s in stats], index=months)
rates = pd.DataFrame({split: problems[SPLITS[split]].sum() / monthly.rows_raw[SPLITS[split]].sum()
                      for split in ("train", "test")})
print(rates.map("{:.2%}".format))
print()
for split in ("all", "train", "test"):
    m = monthly[SPLITS[split]]
    print(f"{split:5} kept {m.rows_clean.sum():,} of {m.rows_raw.sum():,} trips ({m.rows_clean.sum() / m.rows_raw.sum():.1%})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 3.5))
monthly.kept.mul(100).plot(ax=axes[0], marker="o", title="% of trips kept after cleaning")
axes[0].axvline(months.index(TEST_START), color="grey", ls="--")
axes[0].set_xticks(range(0, len(months), 6), months[::6])

problems.div(monthly.rows_raw, axis=0).mul(100).plot(ax=axes[1], title="% of trips hitting each rule")
axes[1].axvline(months.index(TEST_START), color="grey", ls="--")
axes[1].set_xticks(range(0, len(months), 6), months[::6])
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/02_cleaning.png", bbox_inches="tight")

**Notes:** 
- Cleaning keeps **92.8%** of trips (92.3% train, 94.1% test) 
- Moves a lot from month to month: about 95-96% (2024), to 84% in Nov 2025, then back to 93% in Dec 2025.

- Almost all of that comes from the `fare < $3` rule. This drops 1-2% of trips in 2024, 4-12% in 2025, then under 1% from Dec 2025 onwards. 
- Zero-distance and sub-1-minute trips roughly double between 2024 and 2026.

## 3. Trip duration (the target)

In [ ]:
target = {}
for split in ("train", "test"):
    h = total("hist_rate", split).sum(0)
    n, s, ss = total("by_hour", split).sum(1)
    target[split] = {"trips": n, "mean": s / n, "std": np.sqrt(ss / n - (s / n) ** 2),
                     **{f"p{q}": hist_quantile(h, q / 100) for q in (5, 25, 50, 75, 95, 99)}}
    plt.stairs(h / h.sum() / BIN, EDGES, label=f"{split} (median {target[split]['p50']:.1f} min)", lw=1.5)

plt.gcf().set_size_inches(9, 3.5)
plt.xlim(0, 90)
plt.legend()
plt.xlabel("minutes (cut off at 90)"); plt.ylabel("share of trips per minute")
plt.title("Trip duration")
plt.savefig(f"{FIG_DIR}/03_target.png", bbox_inches="tight")
pd.DataFrame(target).round(2)

**Notes:** 
- Most trips are short (train median 13.4 min, mean 17.1). 
- Test trips are about a minute longer through the middle (median 14.3).
- The tail will push RMSE up, so MAE is worth looking at too.

## 4. Time of day and day of week

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))

for split, ls in (("train", "-"), ("test", "--")):
    h = total("hist_day_hour", split)
    for name, days in (("weekdays", slice(0, 5)), ("weekends", slice(5, 7))):
        by_hour = [hist_quantile(h[days, hour].sum(0)) for hour in range(24)]
        axes[0].plot(by_hour, ls=ls, marker="o", ms=3, label=f"{name} ({split})",
                     color="tab:blue" if name == "weekdays" else "tab:orange")
axes[0].set(title="Median duration by hour", xlabel="pickup hour", ylabel="minutes")
axes[0].legend(fontsize=8)

h = total("hist_day_hour", "train")
by_day = np.array([hist_quantile(h[day].sum(0)) for day in range(7)])
axes[1].bar(DAYS, by_day, color=["tab:blue"] * 5 + ["tab:orange"] * 2)
axes[1].set(title="Median duration by day (train)", ylabel="minutes", ylim=(by_day.min() - 1, by_day.max() + 0.5))

counts = h.sum((0, 2)) / len(np.flatnonzero(SPLITS["train"])) / 1000
axes[2].fill_between(range(24), counts, alpha=0.4)
axes[2].plot(counts)
axes[2].set(title="Trips by pickup hour (train, per month)", xlabel="pickup hour", ylabel="thousand trips")

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/04_time.png", bbox_inches="tight")

In [ ]:
heat = np.apply_along_axis(hist_quantile, 2, total("hist_day_hour", "train"))
plt.figure(figsize=(12, 3))
plt.imshow(heat, aspect="auto", cmap="magma")
plt.colorbar(label="minutes")
plt.yticks(range(7), DAYS)
plt.xticks(range(24))
plt.grid(False)
plt.xlabel("pickup hour")
plt.title("Median duration by weekday and hour (train)")
plt.savefig(f"{FIG_DIR}/04_heatmap.png", bbox_inches="tight")

**Notes:** 
- Weekday trips are slowest from about 11am to 3pm (about 15 min). 
- Weekend mornings are quickest (about 10.4 min). 
- Trips spike at 4-5am
- Thursday is the slowest day and Sunday the quickest, with a gap of ~1 minute. 
- The busiest hour is 6pm.
- Test lines (dashed) have the same shape but are about a minute higher at almost every hour.

## 5. Trip destinations

In [ ]:
rate_table = {}
for split in ("train", "test"):
    h = total("hist_rate", split)
    rate_table[f"share {split}"] = h.sum(1) / h.sum()
    rate_table[f"median {split}"] = [hist_quantile(row) if row.sum() else np.nan for row in h]
rate_table = pd.DataFrame(rate_table, index=[RATE_NAMES[c] for c in RATE_CODES]).sort_values("median train")

rate_table["median train"].plot.barh(figsize=(7, 3.5), color="tab:purple", title="Median trip duration by rate code (train)")
plt.xlabel("minutes"); plt.ylabel("")
plt.savefig(f"{FIG_DIR}/05_rate_codes.png", bbox_inches="tight")
rate_table.style.format({"share train": "{:.2%}", "share test": "{:.2%}", "median train": "{:.1f}", "median test": "{:.1f}"})

In [ ]:
# rate code mix over time
mix = pd.DataFrame([s["hist_rate"].sum(1) / s["rows_clean"] for s in stats], index=months,
                   columns=[RATE_NAMES[c] for c in RATE_CODES])
ax = mix[["missing", "JFK", "Newark", "unknown", "negotiated"]].mul(100).plot(figsize=(12, 3.5), marker="o", ms=3,
                                                                             title="% of cleaned trips by rate code")
ax.axvline(months.index(TEST_START), color="grey", ls="--")
ax.set_xticks(range(0, len(months), 6), months[::6])
plt.savefig(f"{FIG_DIR}/05_rate_mix.png", bbox_inches="tight")

In [ ]:
# most popular routes, all months
n, s, _ = total("by_route")
top = np.argsort(n)[::-1][:10]
pd.DataFrame({"PULocationID": top // 266, "DOLocationID": top % 266, "trips": n[top],
              "share": n[top] / n.sum(), "mean duration": s[top] / n[top]}).round(3)

In [ ]:
# how much of the variation in duration can be explained by guessing groups average
def r2_out_of_sample(groups):
    n, s, _ = total(groups, "train")
    mean = np.where(n > 0, s / np.maximum(n, 1), s.sum() / n.sum())
    n_t, s_t, ss_t = total(groups, "test")
    sse = (ss_t - 2 * mean * s_t + n_t * mean ** 2).sum()
    return 1 - sse / (ss_t.sum() - s_t.sum() ** 2 / n_t.sum())

groupings = {"hour only": "by_hour", "rate code only": "by_rate", "pickup zone": "by_pickup", "route (PU, DO)": "by_route"}
pd.DataFrame({"train": {k: r2(g) for k, g in groupings.items()},
              "train -> test": {k: r2_out_of_sample(g) for k, g in groupings.items()}}).round(3)

**Notes:** 
- Trips with a missing rate code go from 14.5% of train to 25.1% of test, and standard trips fall from 80% to 69%. 
- Missing-code trips run longer than standard ones (median 16-17 vs 12 min)
- The `unknown` (99) code almost disappears in 2026-06, the same month `request_source` appears. 
- JFK trips take about 51 min against 12 for standard ones.

- Route alone explains 71% of the variation on train, and 69% when train averages are used to predict 2026.
- Rate code drops more (0.33 to 0.27)
- Hour alone explains under 2%.

## 6. Distance

In [ ]:
# 3,000 random trips from each training month
sample = pd.concat(s["sample"] for s, keep in zip(stats, SPLITS["train"]) if keep)
sample["rate"] = sample.RatecodeID.fillna(0).map(RATE_NAMES)

# colour the airport trips so they stand out
plt.figure(figsize=(8, 5))
for name, colour in [("standard", "tab:blue"), ("JFK", "tab:red"), ("Newark", "tab:green")]:
    d = sample[sample.rate == name]
    plt.scatter(d.trip_distance, d.duration_min, s=2, alpha=0.3, color=colour, label=name)
plt.xlim(0, 40); plt.ylim(0, 120)
plt.xlabel("trip distance (miles)"); plt.ylabel("duration (minutes)")
plt.title(f"Distance vs duration ({len(sample):,} trips sampled from train)")
plt.legend(markerscale=6)
plt.savefig(f"{FIG_DIR}/06_distance.png", bbox_inches="tight")


def correlations(split):
    # exact correlation matrix from n, column sums and X^T X
    n, s, xtx = (sum(part) for part in zip(*(st["corr"] for st, keep in zip(stats, SPLITS[split]) if keep)))
    mean = s / n
    cov = xtx / n - np.outer(mean, mean)
    sd = np.sqrt(np.diag(cov))
    return pd.DataFrame(cov / np.outer(sd, sd), index=NUMERIC, columns=NUMERIC)

corr = {split: correlations(split) for split in ("train", "test")}
corr["train"].loc[["trip_distance", "fare_amount", "duration_min"], ["trip_distance", "fare_amount", "duration_min"]].round(3)

**Notes:** 
- Across all of train, distance and fare are strongly correlated with duration (0.79 and 0.83) (can't be used, as unknown until trip is over)

## 7. Changes over time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for year, colour in YEAR_COLOURS.items():
    d = monthly[monthly.index.str.startswith(year)]
    label = year + (" (test)" if year >= TEST_START[:4] else "")
    x = d.index.str[5:].astype(int)
    axes[0].plot(x, d.mean_duration, marker="o", color=colour, label=label, lw=2)
    axes[1].plot(x, d.rows_clean / 1e6, marker="o", color=colour, label=label, lw=2)
axes[0].set(title="Mean trip duration", ylabel="minutes")
axes[1].set(title="Cleaned trips per month", ylabel="millions")
for ax in axes:
    ax.set_xticks(range(1, 13), MONTHS)
    ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/07_over_time.png", bbox_inches="tight")

n, s, _ = np.array([s["by_hour"].sum(1) for s in stats]).T
pd.DataFrame({"trips": n, "s": s}, index=months).groupby(lambda m: m[:4]).sum().eval("mean_duration = s / trips").drop(columns="s")

In [ ]:
jan_sep = monthly[monthly.index.str[5:] <= "09"]
jan_sep.groupby(jan_sep.index.str[:4])[["mean_duration", "median_duration", "kept"]].mean().round(3)

**Notes:** 
- Yearly average goes 17.0, 17.2, 17.8 min
- Comparing only Jan-Sep gives 16.6, 16.7, 17.8
- 2024 and 2025 are almost the same, increase comes in 2026. 
- January is always quickest, and May and Sep-Dec are slowest. 
- Models will likely guess low on early 2026 as train/test split is by time.

## 8. The actual model features

In [ ]:
# correlation of each numeric feature with duration, over every trip in each split
pd.DataFrame({split: c["duration_min"].drop(["duration_min", "trip_distance", "fare_amount"])
              for split, c in corr.items()}).round(3)

**Notes:** 
- Features barely correlate with duration. 
- Strongest is `hour_cos` at about -0.10, the rest are under 0.05. 
- Month features flip between train and test (`month_sin` -0.05 vs 0.00), partly because test only covers Jan-Sep.

## 9. Test vs training sets

Models learn from 2024-2025 and get scored on 2026

In [ ]:
def share(key, split):
    n = total(key, split)[0]
    return n / n.sum()

# total variation distance
shift = {name: 0.5 * np.abs(share(key, "train") - share(key, "test")).sum()
         for name, key in [("pickup hour", "by_hour"), ("rate code", "by_rate"),
                           ("pickup zone", "by_pickup"), ("route", "by_route")]}
print("how different is test from train (0 = identical, 1 = no overlap):")
for name, value in shift.items():
    print(f"  {name:12} {value:.3f}")

n_train, n_test = total("by_route", "train")[0], total("by_route", "test")[0]
print()
print(f"routes seen in train: {(n_train > 0).sum():,}, in test: {(n_test > 0).sum():,}")
for k in (1, 30):
    rare = n_train < k
    print(f"test trips on a route with < {k} train trips: {n_test[rare].sum() / n_test.sum():.3%}")

In [ ]:
# pickup zones that grew or shrank the most between train and test
zones = pd.DataFrame({"train": share("by_pickup", "train"), "test": share("by_pickup", "test")})
zones["change"] = zones.test - zones.train
moved = zones.loc[zones.change.abs().sort_values(ascending=False).index[:12]].sort_values("change")
moved.change.mul(100).plot.barh(figsize=(7, 4), color=np.where(moved.change > 0, "tab:orange", "tab:blue"),
                                title="Biggest changes in pickup zone share, test minus train")
plt.xlabel("percentage points"); plt.ylabel("PULocationID")
plt.savefig(f"{FIG_DIR}/09_zone_shift.png", bbox_inches="tight")
moved.mul(100).round(2)

**Notes:** 
- Only 0.009% of 2026 trips are on a route never seen in training
- 0.5% are on routes with fewer than 30 training trips
- The pickup hour and zone mixes barely move (0.03 and 0.05). 
- Biggest shift is rate code (0.12), which is the growth in missing `RatecodeID`.  
- Zone changes are small: JFK (132), LaGuardia (138) and Midtown (161, 162, 163, 230) each lose 0.3-0.5 points, and the Lower East Side and East Village (4, 79, 148) gain a little.

## Summary

- There are 123.5M trips. Cleaning keeps 92.8%, mostly because of the `fare < $3` rule.
- Missing `RatecodeID` grows from about 10% of trips in 2024 to about 27% in 2026. These trips run longer, and this is the biggest difference between train and test.
- The route alone explains about 70% of the variation, and still holds when 2024-25 averages are used on 2026. Almost no test trips are on routes the training data hasn't seen.
- Trips in 2026 are about a minute longer, mostly in Jan-May. The daily and weekly patterns keep the same shape.
- Duration has a long right tail (median 13.4 min, mean 17.1, p99 about 71 min), so RMSE will be dominated by a few long trips.
- Time features on their own are weak.
- Distance and fare correlate strongly with duration (0.79 and 0.83) but aren't known until the trip ends, so they're left out.

**Implications for modelling**
- Pickup and dropoff zone should be the core features.
- Tree models should do better than linear, because duration depends on combinations of features.
- Report MAE alongside RMSE because of the long tail.
- Models are expected to slightly under-predict on 2026, since 2026 trips are about a minute longer than 2024-25 ones on average.
- Treating missing `RatecodeID` as its own category (0) is important, because its share nearly doubles in test.


# Data Preprocessing

Data cleaning and preprocessing

Loads the Parquet files efficiently and cleans them before any modelling. Processed data is saved.

## Data Parallelism

Data from input files is read into smmaller shards. Shards are processed in parallel using multiple processing before being merged together (data parallelism). Preprocessing includes cleaning and feature extraction.

## Task Parallelism #1 - Feature Extraction

Feature extraction is done in parallel using multiple threads (task parallelism). Different columns get processed using multiple threads before being merged together.

## Non-Deterministic #1 - THEORY

In the **process_parallel()** the common way to consume ProcessPoolExecutor results would be using `concurrent.futures.as_completed()`, which yields futures in the order they finish, not the order they were submitted. The completion order depends on OS scheduling, therefore queue is used instead to ensure a deterministic outcome.

In [ ]:
%%writefile src/preprocess.py
# preprocess.py - Igor Kochanski 23358459

# Files are read and sharded into equal smaller chunks
# Shards are multi-processed in parallel, then merged together (data parallelism)
# Preprocessing includes cleaning and feature extraction
# Feature extraction is multi-threaded (task parallelism)
# Worker 1 = sequential processing, Worker N = parallel processing
# Results are verified to be identical between sequential and parallel
# Work is done saved to a temporary directory and done in batches to reduce memory usage

import glob
import os
import csv
import argparse
import tempfile
from collections import deque
from concurrent.futures import ThreadPoolExecutor # threads for feature extraction
from concurrent.futures import ProcessPoolExecutor # processes for sharding
from concurrent.futures import as_completed
from pathlib import Path
from tqdm import tqdm
from time import perf_counter

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA_DIR = "./data"
PROCESSED_DIR = "./data/processed"
RESULTS_DIR = "./results/preprocess"

SHARD_ROWS = 500_000 # rows per shard
WORKERS = 8 # number of processes for sharding
THREADS = 4 # number of threads for feature extraction

NON_DETERMINISTIC = False # if True, results may come back in a different order than the input files

# months before TEST_START are training data, months after TEST_END are ignored
TEST_START, TEST_END = "2026-01", "2026-09"

PICKUP, DROPOFF = "tpep_pickup_datetime", "tpep_dropoff_datetime"
COLUMNS = [
    PICKUP,
    DROPOFF,
    "trip_distance",
    "fare_amount",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
]

# Reasonable bounds for NYC taxi trips (same thresholds as example parallel_features.py).
MIN_DURATION, MAX_DURATION = 1, 180      # minutes
MIN_DISTANCE, MAX_DISTANCE = 0.1, 60     # miles
MIN_FARE, MAX_FARE = 3, 250              # dollars

# Fixed categories keep the train and test encodings identical.
ZONES = range(1, 266)                    # TLC taxi zone IDs
RATE_CODES = [0, 1, 2, 3, 4, 5, 6, 99]   # 0 = Flex Fare (null RatecodeID)

FEATURES = [
    "PULocationID",
    "DOLocationID",
    "RatecodeID",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "month_sin",
    "month_cos",
    "is_weekend",
]


def month_of(path):
    # '.../yellow_tripdata_2026-03.parquet' -> '2026-03'
    return Path(path).stem.removeprefix("yellow_tripdata_")


def cyclic(values, period, name):
    # periodic feature encoding
    angle = 2 * np.pi * values / period
    return {f"{name}_sin": np.sin(angle), f"{name}_cos": np.cos(angle)}


def extract_features(df, threads=4):
    # task parallelism for feature extraction
    pickup = pd.to_datetime(df[PICKUP])

    hour = pickup.dt.hour
    weekday = pickup.dt.day_of_week
    month = pickup.dt.month
    year = pickup.dt.year

    tasks = [
        lambda: {
            "PULocationID": pd.Categorical(df["PULocationID"], categories=ZONES),
            "DOLocationID": pd.Categorical(df["DOLocationID"], categories=ZONES),
            "RatecodeID": pd.Categorical(df["RatecodeID"].fillna(0), categories=RATE_CODES),
        },
        lambda: {**cyclic(hour, 24, "hour")},
        lambda: {**cyclic(weekday, 7, "weekday"), "is_weekend": (weekday >= 5).astype(int)},
        lambda: {**cyclic(month - 1, 12, "month")},
    ]

    with ThreadPoolExecutor(threads) as executor:
        parts = executor.map(lambda task: task(), tasks)

    columns = {name: col for part in parts for name, col in part.items()}

    # year is kept for sampling, not a model feature
    return pd.DataFrame({**columns, "year": year}, index=df.index)[FEATURES + ["year"]]


def clean(df, path):
    # keep valid trips only, add duration_min column
    start = pd.Timestamp(f"{month_of(path)}-01")
    end = start + pd.DateOffset(months=1)

    pickup = pd.to_datetime(df[PICKUP])
    dropoff = pd.to_datetime(df[DROPOFF])
    duration = (dropoff - pickup).dt.total_seconds() / 60

    # no spill-over pickups, nulls, zero/negative durations, zero-distance trips, negative fares, or meter errors
    ok = (
        (pickup >= start)
        & (pickup < end)
        & duration.between(MIN_DURATION, MAX_DURATION)
        & df["trip_distance"].between(MIN_DISTANCE, MAX_DISTANCE)
        & df["fare_amount"].between(MIN_FARE, MAX_FARE)
        & df["RatecodeID"].fillna(0).isin(RATE_CODES)
    )
    return df[ok].assign(duration_min=duration[ok])


def iter_shards(files):
    # yield fixed-size (shard_id, path, batch) shards, in file order
    shard_id = 0
    for path in files:
        for batch in pq.ParquetFile(path).iter_batches(batch_size=SHARD_ROWS, columns=COLUMNS, use_threads=False):
            yield shard_id, path, batch
            shard_id += 1

def process_shard(shard_id, path, batch, temp_dir):
    # clean one shard and write it to temp_dir, then return (file, rows_in, rows_out)
    df = clean(batch.to_pandas(), path)
    features = extract_features(df, threads=THREADS).assign(duration_min=df["duration_min"])

    out = os.path.join(temp_dir, f"shard_{shard_id:03d}.parquet")
    features.to_parquet(out, index=False)
    return out, batch.num_rows, len(df)


# DETERMINISTIC: results come back in shard order, not completion order
def process_parallel(shards, workers, temp_dir):

    if NON_DETERMINISTIC:
        return process_parallel_non_deterministic(shards, workers, temp_dir)

    # process shards in parallel, results come back in shard order
    results, pending = [], deque() # queue instead of as_completed() to preserve order of shards

    with ProcessPoolExecutor(max_workers=workers) as executor:
        for shard in tqdm(shards, desc="Parallel", unit=" shard", leave=False):
            pending.append(executor.submit(process_shard, *shard, temp_dir))
            if len(pending) >= workers:
                results.append(pending.popleft().result())
        results.extend(future.result() for future in pending)

    return results


# NON-DETERMINISTIC: results come back in completion order, not shard order
def process_parallel_non_deterministic(shards, workers, temp_dir):
    results = []

    with ProcessPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(process_shard, *shard, temp_dir) for shard in shards]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Parallel", unit=" shard", leave=False):
            results.append(future.result())

    return results


def merge_shards(shard_paths, output_path):
    # append the shards to one Parquet file, one shard in memory at a time
    schema = pq.ParquetFile(shard_paths[0]).schema_arrow
    
    with pq.ParquetWriter(output_path, schema) as writer:
        for path in tqdm(shard_paths, desc="Merging", unit=" shard", leave=False):
            writer.write_table(pq.ParquetFile(path).read())


def assert_parquet_equal(left_path, right_path):
    # compare in batches, so both full frames never coexist in memory
    left, right = pq.ParquetFile(left_path), pq.ParquetFile(right_path)

    if left.metadata.num_rows != right.metadata.num_rows:
        raise AssertionError(f"row count differs: {left.metadata.num_rows} != {right.metadata.num_rows}")
    if not left.schema_arrow.equals(right.schema_arrow):
        raise AssertionError("Parquet schemas differ")

    # same total rows, so zip() cannot silently drop non-empty batches
    for left_batch, right_batch in zip(
        left.iter_batches(batch_size=SHARD_ROWS, use_threads=False),
        right.iter_batches(batch_size=SHARD_ROWS, use_threads=False),
    ):
        pd.testing.assert_frame_equal(left_batch.to_pandas(), right_batch.to_pandas())


def run(files, workers, output_path, shard_dir):
    # process fixed-size shards and merge them into output, returns (n_shards, rows_before, rows_after)
    os.makedirs(shard_dir, exist_ok=True)

    shards = iter_shards(files)
    shards = list(shards) # materialize iterator for progress bar

    if workers == 1: # sequential processing
        results = [process_shard(*shard, shard_dir) for shard in tqdm(shards, desc="Sequential", unit=" shard", leave=False)]
    else: # parallel processing
        results = process_parallel(shards, workers, shard_dir)

    merge_shards([out for out, _, _ in results], output_path)

    return (
        len(results),
        sum(rows_in for _, rows_in, _ in results),
        sum(rows_out for _, _, rows_out in results),
    )


def save_summary(split, shard_rows, workers, threads, sequential, parallel):
    # save summary of run to CSV
    os.makedirs(RESULTS_DIR, exist_ok=True)

    csv_path = f"{RESULTS_DIR}/summary.csv"
    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)

        if os.path.getsize(csv_path) == 0:
            writer.writerow(["split", "shard_rows", "workers", "threads", "sequential", "parallel"])
        writer.writerow([split, shard_rows, workers, threads, sequential, parallel])


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--shard-rows", type=int, default=SHARD_ROWS)
    parser.add_argument("--workers", type=int, default=WORKERS)
    parser.add_argument("--threads", type=int, default=THREADS)
    parser.add_argument("--non-deterministic", action="store_true", default=NON_DETERMINISTIC)
    args = parser.parse_args()

    SHARD_ROWS = args.shard_rows
    WORKERS = args.workers
    THREADS = args.threads
    NON_DETERMINISTIC = args.non_deterministic

    os.makedirs(PROCESSED_DIR, exist_ok=True)

    files = sorted(glob.glob(os.path.join(DATA_DIR, "yellow_tripdata_*.parquet")))

    # time-based split
    splits = {
        "train": [p for p in files if month_of(p) < TEST_START],
        "test": [p for p in files if TEST_START <= month_of(p) <= TEST_END]
    }

    # run the preprocessing tasks in a temporary directory, then move the final Parquet files to PROCESSED_DIR
    with tempfile.TemporaryDirectory(prefix="preprocess_", dir=PROCESSED_DIR) as tmp:
        for split, paths in splits.items():
            started = perf_counter()  # start timer for this split
            
            print(f"{split} split: {month_of(paths[0])} to {month_of(paths[-1])}")
 
            # Run once sequentially and once in parallel
            output, seconds = {}, {}
 
            for mode, workers in (("seq", 1), ("par", WORKERS)):
                output[mode] = os.path.join(tmp, f"{split}_{mode}.parquet")
                mode_started = perf_counter()
                stats = run(paths, workers, output[mode], os.path.join(tmp, f"{split}_{mode}"))
                seconds[mode] = perf_counter() - mode_started
 
            n_shards, rows_before, rows_after = stats

            print(f"  {len(paths)} files, {n_shards} shards")
            print(f"  {rows_before:,} -> {rows_after:,} trips")
            print(f"  1 workers, {THREADS} threads = {seconds['seq']:.1f} s")
            print(f"  {WORKERS} workers, {THREADS} threads = {seconds['par']:.1f} s")
            print(f"  speedup: {seconds['seq'] / seconds['par']:.1f}x")
            
            # check sequential and parallel outputs are identical
            assert_parquet_equal(output["seq"], output["par"])
            print("  verified outputs match")

            # move final Parquet file to PROCESSED_DIR
            os.replace(output["par"], os.path.join(PROCESSED_DIR, f"{split}.parquet"))
            print(f"  saved {split}.parquet in {perf_counter() - started:.1f} s")

            # save summary of run
            save_summary(split, SHARD_ROWS, WORKERS, THREADS, seconds["seq"], seconds["par"])

In [ ]:
%%writefile src/benchmark.py
# Benchmark scripts with different configurations

import sys
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC_DIR = "./src/"
RESULTS_DIR = "./results/"

def benchmark_preprocess(configs, attempts=1):
    # run the preprocess.py with different configurations and log the outputs
    script = f"{SRC_DIR}/preprocess.py"
    output_dir = f"{RESULTS_DIR}/preprocess"
    os.makedirs(output_dir, exist_ok=True)

    run = 1
    for shard_rows, workers, threads in configs:
        for _ in range(attempts):
            with open(f"{output_dir}/run{run:03d}-{shard_rows}-{workers}-{threads}.log", "w") as log:
                process = subprocess.Popen(
                    [sys.executable, "-u", script,
                    "--shard-rows", str(shard_rows),
                    "--workers", str(workers),
                    "--threads", str(threads)],
                    stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT,
                    text=True,
                    bufsize=1
                )

                # Print and log output while preprocessing runs
                for line in process.stdout:
                    #print(line, end="", flush=True) # print in terminal
                    log.write(line)
                    log.flush()

                return_code = process.wait()

                if return_code != 0:
                    raise subprocess.CalledProcessError(return_code, process.args)

            print(f"Finished run {run:03d}: {shard_rows} shard rows, {workers} workers, {threads} threads")
            run += 1


def show_non_deterministic(configs, attempts=1):
    # run the preprocess.py with different configurations and log the outputs
    script = f"{SRC_DIR}/preprocess.py"
    output_dir = f"{RESULTS_DIR}/preprocess"
    os.makedirs(output_dir, exist_ok=True)

    run = 1
    return_code = 0
    for shard_rows, workers, threads in configs:
        for _ in range(attempts):
            try:
                output_lines = []
                with open(f"{output_dir}/run{run:03d}-{shard_rows}-{workers}-{threads}-nondet.log", "w") as log:
                    process = subprocess.Popen(
                        [sys.executable, "-u", script,
                        "--shard-rows", str(shard_rows),
                        "--workers", str(workers),
                        "--threads", str(threads),
                        "--non-deterministic"],
                        stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT,
                        text=True,
                        bufsize=1
                    )

                    # Print and log output while preprocessing runs
                    for line in process.stdout:
                        #print(line, end="", flush=True) # print in terminal
                        log.write(line)
                        log.flush()
                        output_lines.append(line)

                    return_code = process.wait()

                if return_code != 0:
                    # Include the tail of preprocess.py's output in the error
                    tail = "".join(output_lines[-50:])
                    raise RuntimeError(
                        f"preprocess.py exited with code {return_code} "
                        f"(shard_rows={shard_rows}, workers={workers}, threads={threads}).\n"
                        f"Last output:\n{tail}"
                    )

                print(f"Finished run {run:03d}: {shard_rows} shard rows, {workers} workers, {threads} threads")
            except Exception as e:
                return e
            run += 1
    return None


def plot_preprocess_results():
    df = pd.read_csv(f"{RESULTS_DIR}/preprocess/summary.csv")

    avg = df.groupby(["split", "shard_rows", "workers", "threads"])[["sequential", "parallel"]].mean().reset_index()

    avg["speedup"] = avg["sequential"] / avg["parallel"]

    configs = avg[["shard_rows", "workers", "threads"]].drop_duplicates()
    labels = [f"{r.shard_rows:,}\n{r.workers} workers\n{r.threads} threads" for r in configs.itertuples(index=False)]

    x = np.arange(len(configs))
    width = 0.35

    fig, axes = plt.subplots(1, avg["split"].nunique(), figsize=(7 * avg["split"].nunique(), 5), dpi=180)
    axes = np.atleast_1d(axes)

    for ax, (split, data) in zip(axes, avg.groupby("split")):
        data = data.sort_values(["shard_rows", "workers", "threads"])

        ax.bar(x - width / 2, data["sequential"], width, label="Sequential")
        ax.bar(x + width / 2, data["parallel"], width, label="Parallel")

        for i, row in enumerate(data.itertuples()):
            ax.text(x[i], max(row.sequential, row.parallel) * 1.01, f"{row.speedup:.2f}x", ha="center", va="bottom", fontweight="bold")

        ax.set(title=f"{split.capitalize()} preprocessing", xlabel="Configuration", ylabel="Average execution time (seconds)", xticks=x, xticklabels=labels)
        ax.legend()
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle("Average Sequential vs Parallel Preprocessing Time")
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/preprocess/benchmark.png",dpi=300, bbox_inches="tight")
    plt.show()

    print("\nAverage execution times and speedups:")
    print(avg.to_string(index=False))


if __name__ == "__main__":

    #shard_rows, workers, threads
    preprocess_configs = [
        # Small shards
        (100_000, 2, 4),
        (100_000, 4, 4),
        (100_000, 8, 4),

        # Large shards
        (1_000_000, 2, 4),
        (1_000_000, 4, 4),
        (1_000_000, 8, 4),
    ]
    benchmark_preprocess(preprocess_configs, attempts=3)
    plot_preprocess_results()

## Preprocessing comparison of sequential and parallel execution

Each parallel configuration is ran with the against a sequential run on the same data with times recorded. The two are compared to ensure no differences, if a difference was to occur, an error would be thrown. Multiple attempts of each config run are averaged for consistent resuts.

Chart is displayed showing speedup vs workers

Discussion using Amdahl's law: estimated parallel fraction p and overheads

In [ ]:
from src.benchmark import benchmark_preprocess, plot_preprocess_results

#shard_rows, workers, threads
preprocess_configs = [
    # Small shards
    (100_000, 2, 4),
    (100_000, 4, 4),
    (100_000, 8, 4),

    # Large shards
    (1_000_000, 2, 4),
    (1_000_000, 4, 4),
    (1_000_000, 8, 4),
]

benchmark_preprocess(preprocess_configs, attempts=5)
plot_preprocess_results()

## Non-Deterministic #1 - EVIDENCE

Running the same `preprocess.py` non-deterministically fails the comparison check, throwing and error. The error is caught and printed. This is because rows don't return in the same order.

In [ ]:
from src.benchmark import show_non_deterministic

preprocess_configs = [(1_000_000, 4, 4)]

show_non_deterministic(preprocess_configs, attempts=1)

## Task Parallelism #2 - Hyperparameter Search

Every (model,hyperparameters) setting in `GRID` is one **trial**: 2 Ridge models (one-hot zones and rate codes) and 6 hist gradient boosting models (learning rate 0.3/0.5/0.7 x 200/500 iterations).

**why the trials are independent:** each one trains on the same 2024 sample and is scored on the same 2025 validation sample,shares no state, and returnss one row of scores. So each trial runs as its own task in a `ProcessPoolExecutor`, processes not threads since training is CPU-bound and threads would be limited by the GIL.

Design choices:
- **Selection on validation only**: the best setting is chosen on 2025 validation RMSE;the 2026 test set is never used here.
- **Workers load the data themselves** from a cached sample file, so the main process never pickles millions of rows into every task.
- **One maths thread per process** (env variables set before numpy/sklearn are imported),so N processes do not oversubscribe the cores.
- **`pool.map` + sort by trial id**: results come back in submission order, so the table is identical for any worker count.
- **`--order longest_first`** submits the slowest trials first to reduce load imbalance.

In [ ]:
%%writefile src/search.py
#src/search.py
# Tudor Ventuneac - 23390131
#hyperparameter search for task parallelism
#evey (model,hyperparemeters) setting in GRID is 1 trial
#trials are independent so same training data, but different config 
# so nothings shared while they run so each trial runs as its own process

# best setting is chosen on the 2025 validation sample


import os

#one maths thread per process set before before numpy or sklearn are imported
#otherwise N processes x M library threads wud fight over the same cores
#could lead to a non deterministic outcome

#added veclib max threads since im on macos and it covers accelerate BLAS used by numpy for macos, should be safe for windows too
for var in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","VECLIB_MAXIMUM_THREADS","MKL_NUM_THREADS"):
    os.environ[var]= "1"

import argparse
import json
from concurrent.futures import ProcessPoolExecutor
from time import perf_counter

import joblib
import numpy
import pandas as pd
import pyarrow.parquet as pq
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error,root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from preprocess import FEATURES, RATE_CODES, ZONES

#Config
PROCESSED_DIR = "./data/processed"
SEARCH_DIR = "./data/processed/search"
MODEL_DIR= "./models"
RESULTS_DIR = "./results"

SEED = 0
TRAIN_FRAC = 0.02 # using 2% of 2024 rows for the search
VAL_FRAC = 0.02 # 2% of 2025 rows used to score each trial
WORKERS = 4 #we'll change this when running on the rockhill servers to 3
REPEATS= 1 #timed runs per worker count since rubric mentions wanting the mean of 3+ for the benchmark
TARGET = "duration_min"
CATEGORICAL = ["PULocationID","DOLocationID","RatecodeID"]
NUMERIC = [f for f in FEATURES if f not in CATEGORICAL]

#2 ridge trials(alpha barelly mattered in the first search) and 6 gb trials
#first search had its best result on the edge of the grid(highest lr,no depth limit)
# so the grid now goes past it: higher learning rates and more boosting iterations
GRID = (
    [("ridge",{"alpha": a}) for a in (0.1, 10.0)]
    + [("hgb", {"learning_rate": lr,"max_iter":it, "max_depth":None})
       for lr in (0.3,0.5,0.7) for it in (200,500)]
)

def expected_cost(name,params):
    #rough relative runn time of a trial which we'll use to submit the slowest one first
    #a 500 iter hgb trial takes about 2.5 times a 200 iter one, ridgei s the quickest
    return params.get("max_iter",0) if name == "hgb" else 0
 


def make_model(name,params):
    #building an unfitted pipeline for 1 trial
    if name == "ridge":
        #linear model so the zones and rate codes have to be one-hot
        categories = [[float(z) for z in ZONES]] * 2 +[[float(r) for r in RATE_CODES]]
        prep = ColumnTransformer([
            ("onehot",OneHotEncoder(categories=categories, handle_unknown="ignore"), CATEGORICAL),
            ("scale", StandardScaler(), NUMERIC),
        ])
        return Pipeline([("prep",prep), ("model",Ridge(**params))])

    #trees can split on the zone id ranges so dont need one-hot
    #fixed iterations and no early stopping so no random val split inside
    if name == "hgb":
        return Pipeline([("model",HistGradientBoostingRegressor(**params, early_stopping=False,random_state=SEED))])

    raise ValueError(f"unknown model {name}")


#writing a seeded random sample of one years rows of train.parquet to search_dir once and returning its path
def prepare_sample(year,frac):
    os.makedirs(SEARCH_DIR,exist_ok=True)
    out = os.path.join(SEARCH_DIR, f"{year}_{frac}.parquet")
    if os.path.exists(out):
        return out

    parts=[]
    for batch in pq.ParquetFile(os.path.join(PROCESSED_DIR,"train.parquet")).iter_batches(
        batch_size=500_000, columns=FEATURES + [TARGET, "year"], use_threads=False):

        batch = batch.to_pandas()
        year_rows = batch[batch["year"] == year]
        parts.append(year_rows.drop(columns="year").sample(frac=frac,random_state=SEED).astype("float32"))

    pd.concat(parts,ignore_index=True).to_parquet(out,index=False)
    return out



#one search task where itll load the data,fit it,score on val and run inside a worker process
#to make sure main process never pickles the worker will read the data from disk 
def run_trial(trial_id,name,params,train_path,val_path):
    started = perf_counter()

    train = pd.read_parquet(train_path)
    val= pd.read_parquet(val_path)

    model=make_model(name,params).fit(train[FEATURES],train[TARGET])
    pred = model.predict(val[FEATURES])

    return {
        "trial": trial_id,
        "model": name,
        "params": json.dumps(params),
        "val_mae": mean_absolute_error(val[TARGET],pred),
        "val_rmse": root_mean_squared_error(val[TARGET],pred),
        "seconds": perf_counter()-started,
        "pid": os.getpid(),
    }


def search(workers,order,train_path,val_path):
    #running every trial in grid
    tasks = list(enumerate(GRID))
    if order == "longest_first":
        #submiting the most expensive trials first so no worker is left with a long trial at the end
        tasks.sort(key=lambda task: expected_cost(*task[1]), reverse=True)

    ids = [trial_id for trial_id,_ in tasks]
    names = [name for _,(name,_) in tasks]
    params = [p for _, (_,p) in tasks]
    n= len(tasks)


    started = perf_counter()
    #sequential baseline
    if workers == 1:
        rows = [run_trial(*args) for args in zip(ids,names,params, [train_path]*n, [val_path] * n)]
    else: #task parallel->ech trial in its own process and the map returns results in submission order
        with ProcessPoolExecutor(max_workers=workers) as pool:
            rows = list(pool.map(run_trial,ids,names,params, [train_path] * n, [val_path] * n))
    wall = perf_counter() - started

     #sorting by trial id so table is the same nomatter the order or worker count
    return pd.DataFrame(rows).sort_values("trial").reset_index(drop=True),wall





def main():
    parser = argparse.ArgumentParser(description="parallel hyperparameter search")
    parser.add_argument("--workers",type=int,nargs="+",default=[WORKERS])
    parser.add_argument("--order",choices=["grid","longest_first"],default="grid")
    parser.add_argument("--repeats",type=int,default=REPEATS,help = "timed runs per worker count")
    args = parser.parse_args()

    train_path = prepare_sample(2024,TRAIN_FRAC)
    val_path =prepare_sample(2025,VAL_FRAC)
    print(f"Train sample: {pq.ParquetFile(train_path).metadata.num_rows:,} rows | "
          f"Validation sample: {pq.ParquetFile(val_path).metadata.num_rows:,} rows")
    print(f"{len(GRID)} trials, order = {args.order},{args.repeats} repeats per worker count\n")

    #warming up via reading both samples once untimed so the first timed run doesnt pay for
    #loadingg thme from the disk while the later runs gets them from the OS file cache
    pd.read_parquet(train_path), pd.read_parquet(val_path)

    os.makedirs(RESULTS_DIR,exist_ok=True)
    metrics = ["trial","model","params","val_mae","val_rmse"]
    reference = None   #scores from the very first run, every later run must match it
    tables, runs = {}, []

    for workers in args.workers:
        per_repeat = []
        for repeat in range(1, args.repeats + 1):
            table,wall = search(workers, args.order,train_path, val_path)

            #regardles of worker count or repeat, same trials have to give the same scores
            if reference is None:
                reference = table[metrics]
            pd.testing.assert_frame_equal(reference, table[metrics])

            #summing all the seconds for total work done by all trials
            busy = table["seconds"].sum()
            runs.append({"workers": workers, "order": args.order, "repeat": repeat,
                         "wall_s": wall, "busy_s": busy})
            per_repeat.append(table)
            print(f"  {workers} workers, run {repeat}: wall clock {wall:.1f} s | "
                  f"sum of trial times {busy:.1f} s")

        #per trial times averaged over the repeats (scores are identical so they're just kept)
        mean_table = per_repeat[0].copy()
        mean_table["seconds"] = sum(t["seconds"] for t in per_repeat) / len(per_repeat)
        tables[workers] = mean_table

        print(f">>> {workers} workers (trial seconds = mean of {args.repeats} run(s))")
        print(mean_table.drop(columns="pid").round(3).to_string(index=False))
        print()
        mean_table.drop(columns="pid").to_csv(
            os.path.join(RESULTS_DIR, f"search_{workers}w_{args.order}.csv"),index=False)

    print("verified: identical validation scores for", ", ".join(map(str,args.workers)),
          f"workers across {args.repeats} repeat(s)")

    #every timed run,+ a summary with mean and spread per worker count
    runs = pd.DataFrame(runs)
    runs.to_csv(os.path.join(RESULTS_DIR, f"search_runs_{args.order}.csv"), index=False)

    summary = runs.groupby("workers").agg(
        wall_mean=("wall_s","mean"), wall_std=("wall_s","std"),
        busy_mean=("busy_s","mean"), runs=("repeat","count")).reset_index()
    if 1 in args.workers:
        base = summary.loc[summary["workers"] == 1, "wall_mean"].iloc[0]
        summary["speedup"] = base / summary["wall_mean"]
        summary["efficiency"] = summary["speedup"] / summary["workers"]
    summary.to_csv(os.path.join(RESULTS_DIR, f"search_timings_{args.order}.csv"), index=False)
    print("\n" + summary.round(3).to_string(index=False))

 
    # pick the best setting on val rmse, refit it, save it for evaluate.py
    best = reference.loc[reference["val_rmse"].idxmin()]
    print(f"\nBest: {best['model']} {best['params']} | val RMSE {best['val_rmse']:.3f} | val MAE {best['val_mae']:.3f}")
 
    train = pd.read_parquet(train_path)
    model = make_model(best["model"], json.loads(best["params"])).fit(train[FEATURES], train[TARGET])
    os.makedirs(MODEL_DIR, exist_ok=True)
    joblib.dump(model, os.path.join(MODEL_DIR, "search_best.joblib"))
    print("Saved models/search_best.joblib (scored on test by evaluate.py)")
 
 
if __name__ == "__main__":
    main()

### Sequential vs parallel: hyperparameter search

the same 8 trials are run with 1 worker(sequential baseline) and with 2 and 3 workers, 3 times each after an untimed warmup. The script stops with an error if any validation score differs between worker counts or repeats. Results are saved to `results/search_*.csv`.

takes about 8mins on a 10-core laptop

In [ ]:
import sys
!{sys.executable} src/search.py --workers 1 2 3 --repeats 3

### Load imbalance

The 500 iteration boosting trials take about 2.5x as long as the 200-iteration ones, and Ridge is quickest. In grid order a long trial can be submitted last and run alone at the end while the other workers sit idle. Submitting the longest trials first (estimated from `max_iter`)lets the short trials fill the gaps.

In [ ]:
!{sys.executable} src/search.py --workers 3 --order longest_first --repeats 3

## Task Parallelism #3 - Gradient Computation

Linear regression for trip duration, trained by batch gradient descent on a 20% sample of 2024 trips(about 7.9M), split into 1M-row shards. Every step is written out below:


1. **Broadcast**: the current weights are sent to every shard task
2. **Compute**: each task returns the *sums* of its trips' gradients and its trip count(not an average, since the shards differ in size, so a mean of means would be wrong)
3. **Combine** (all-reduce part): the main process adds the sums **in shard order** and divides by the total count
4. **Update**: one gradient step on the single shared model

**why the shard tasks are independent**: within a step they al use the same weights and different trips. Shards are saved once as numpy arrays and memory-mapped by the workers. One-hot zones are stored as integer codes; the gradient for a one-hot column is the sum of errors of the trips in that zone (`np.bincount`).

The first run builds the shards in `data/processed/gradient/` and compares against scikit learn Ridge on the same trips.

In [ ]:
%%writefile src/gradient_compute.py
#src/gradient_compute.py
#Tudor Ventuneac - 23390131
#linear regression for trip duration,trained by batch gradient descent
#gonna break it down in terms of broadcasting, compting, combining and then updating
#1 broadcast, current weights are sent to every shard task
#2 compute, each task will return the sums of its trips gradients plus the trip count
#not avg since the shards have diff sizes
#3 combine, main process where itll add the sum in shard order and divde by total count
#4 update, 1 gradient step on the single shared model

import os
for var in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","VECLIB_MAXIMUM_THREADS","MKL_NUM_THREADS"):
    os.environ[var]="1"

import argparse
import json
from concurrent.futures import ProcessPoolExecutor,as_completed
from time import perf_counter
import numpy as np
import pandas as pd
from preprocess import FEATURES,RATE_CODES
from search import CATEGORICAL,SEED,TARGET, make_model, prepare_sample


#config
PREP_DIR= "./data/processed/gradient"
RESULTS_DIR = "./results"

TRAIN_FRAC=0.20 #20% of the 2024 trips, should be around 8mill so each step has enough work to parallelise
VAL_FRAC= 0.02 #same 2025 val sample you changed in search
SHARD_ROWS = 1000000 #1 mill shards per row, equal sized, last one gonna be smaller

ETA=0.2 #lr
LAM=0.0 #l2 reg. strength, not applied to the intercept
STEPS=200 #fixing the number of steps so every run does the exact same amount of work
WORKERS = 4
REPEATS = 1

NUMERIC = [f for f in FEATURES if f not in CATEGORICAL]
N_ZONES= 266 #zone ids 1-265
N_RATES = len(RATE_CODES)


def to_arrays(df):
    #storing the one hot cols as small int codes instead mostly zero cols
    return{
        "num": df[NUMERIC].to_numpy(np.float32),
        "pu": df["PULocationID"].to_numpy().astype(np.int16),
        "do": df["DOLocationID"].to_numpy(np.int16),
        "rate":np.searchsorted(RATE_CODES, df["RatecodeID"].to_numpy()).astype(np.int8),
        "y":df[TARGET].to_numpy(np.float32),
    }

def save_arrays(prefix,arrays):
    for name,values in arrays.items():
        np.save(f"{prefix}_{name}.npy", values)


# memory mapped so the os reads pages on demand an shares them between processes
def load_arrays(prefix):
    return {name:np.load(f"{prefix}_{name}.npy",mmap_mode="r")
            for name in ("num","pu","do","rate","y")}


#sampling once and cutting the training sample into fixed shards and saving val, itll be reused on later runs
def prepare_shards():
    meta_path = os.path.join(PREP_DIR, f"meta_{TRAIN_FRAC}_{SHARD_ROWS}.json")
    if os.path.exists(meta_path):
        return json.load(open(meta_path))

    os.makedirs(PREP_DIR,exist_ok=True)

    train = pd.read_parquet(prepare_sample(2024,TRAIN_FRAC))
    val = pd.read_parquet(prepare_sample(2025,VAL_FRAC))

    rows = []
    for shard_id,start in enumerate(range(0,len(train), SHARD_ROWS)):
        shard = train.iloc[start:start+SHARD_ROWS]
        save_arrays(os.path.join(PREP_DIR,f"s{shard_id:02d}"), to_arrays(shard))
        rows.append(len(shard))
    save_arrays(os.path.join(PREP_DIR,"val"),to_arrays(val))

    meta= {"shard_rows":rows,"train_rows":len(train),"val_rows":len(val)}
    json.dump(meta,open(meta_path,"w"))
    return meta


def init_params():
    #same starting point every run with all weights 0
    return {"num":np.zeros(len(NUMERIC)),"pu":np.zeros(N_ZONES),"do":np.zeros(N_ZONES), "rate":np.zeros(N_RATES),"b":0.0}

#b + numeric weights + one weight per pickup zone,dropoff zone and rate code
def predict(p,d):
    return p["b"] + d["num"] @ p["num"] + p["pu"][d["pu"]] + p["do"][d["do"]] + p["rate"][d["rate"]]


#one task which is summed gradients for 1 shard using the broadcast weights p
#opens its shard from disk via mmap so nothing large get pickled, it calculates each trips error with p nad returns sums plus count
def shard_gradient(shard_id,prep_dir,p):
    d= load_arrays(os.path.join(prep_dir,f"s{shard_id:02d}"))
    #error per trip
    err = predict(p,d) - d["y"]

    return{
        "num": d["num"].T @ err, #sum of error times input
        "pu": np.bincount(d["pu"],weights=err, minlength=N_ZONES), #onehot-> sum of errors per zone
        "do": np.bincount(d["do"], weights=err, minlength=N_ZONES),
        "rate": np.bincount(d["rate"], weights=err, minlength=N_RATES),
        "b": err.sum(), #intercept(sum of errors)
        "sse": err @ err, #for the training rmse
        "n": len(err) #count so we divide once at the end
    }

#adding the shards sum in the order given
def combine(parts):
    #adding the partial sums in the order given(fp addition depends on the order)
    return {key:sum(part[key] for part in parts)for key in parts[0]}

#training loop
#broadcast->compute->combine->divide by total count->1 update
def run_gd(workers,combine_mode,n_shards,steps=STEPS,eta=ETA,lam=LAM):
    #bgd, workers=1 runs the shards one after another in this process
    p,history = init_params(),[]
    ids = list(range(n_shards))
    pool = ProcessPoolExecutor(max_workers=workers) if workers>1 else None

    started = perf_counter()
    try:
        for _ in range(steps):
            if pool is None: #sequential baseline
                parts = [shard_gradient(i,PREP_DIR,p)for i in ids]
            elif combine_mode == "ordered": #map returns results in shared order
                parts = list(pool.map(shard_gradient, ids, [PREP_DIR] * n_shards, [p] * n_shards))
            else:
                futures = [pool.submit(shard_gradient, i, PREP_DIR, p) for i in ids]
                parts = [f.result() for f in as_completed(futures)]

            G= combine(parts)
            N = G["n"]
            for key in ("num","pu","do","rate"):
                p[key] = p[key] - eta * (G[key] / N + lam * p[key])
            p["b"] = p["b"] - eta * G["b"] / N
            #training rmse at the start of this step
            history.append(np.sqrt(G["sse"]/N))

    finally:
        if pool:
            pool.shutdown()
    return p,history,perf_counter() - started

def flat(p):
    return np.concatenate([p["num"],p["pu"],p["do"],p["rate"],[p["b"]]])


def score(p,d):
    err = predict(p,d)-d["y"]
    return float(np.sqrt(np.mean(err ** 2))), float(np.mean(np.abs(err)))



def main():
    parser = argparse.ArgumentParser(description="parallel gradient computation")
    parser.add_argument("--workers", type=int, nargs="+", default=[WORKERS])
    parser.add_argument("--repeats", type=int, default=REPEATS)
    parser.add_argument("--steps", type=int, default=STEPS)
    parser.add_argument("--combine", choices=["ordered", "completion"], default="ordered")
    parser.add_argument("--baseline", action="store_true", help="also fit scikitlearn ridge")
    args = parser.parse_args()

    meta = prepare_shards()
    n_shards = len(meta["shard_rows"])
    print(f"Train: {meta['train_rows']:,} trips in {n_shards} shards {meta['shard_rows']} | "
          f"Validation: {meta['val_rows']:,} trips")
    print(f"{args.steps} steps, eta = {ETA}, lambda = {LAM}, combine = {args.combine}, "
          f"{args.repeats} repeat(s)\n")


    #warmup,one untimed pass so the os has the shard files cached before timing starts
    for i in range(n_shards):
        shard_gradient(i, PREP_DIR, init_params())

    os.makedirs(RESULTS_DIR, exist_ok=True)
    reference, runs = None, []



    for workers in args.workers:
        for repeat in range(1, args.repeats + 1):
            p, history, wall = run_gd(workers, args.combine, n_shards,steps=args.steps)
            weights = flat(p)

            if reference is None:
                reference, best_p, best_history = weights, p, history
            identical = bool(np.array_equal(reference, weights))
            max_diff = float(np.max(np.abs(reference - weights)))

            runs.append({"workers": workers, "combine": args.combine, "repeat": repeat,
                         "wall_s": wall, "step_ms": 1000 * wall /args.steps,
                         "identical": identical, "max_abs_diff": max_diff})
            
            print(f"  {workers} workers, run {repeat}: {wall:.1f} s ({1000 * wall / args.steps:.0f} ms/step) | "
                  f"final train RMSE {history[-1]:.4f} | identical to first run: {identical} "
                  f"(max diff {max_diff:.2e})")


    runs = pd.DataFrame(runs)
    runs.to_csv(os.path.join(RESULTS_DIR, f"gradient_runs_{args.combine}.csv"), index=False)

    summary = runs.groupby("workers").agg(
        wall_mean=("wall_s", "mean"), wall_std=("wall_s", "std"),
        step_ms=("step_ms", "mean"), runs=("repeat", "count"),
        all_identical=("identical", "all")).reset_index()

    if 1 in args.workers:
        base = summary.loc[summary["workers"] == 1, "wall_mean"].iloc[0]
        summary["speedup"] = base / summary["wall_mean"]
        summary["efficiency"] = summary["speedup"] / summary["workers"]
    summary.to_csv(os.path.join(RESULTS_DIR, f"gradient_timings_{args.combine}.csv"), index=False)
    print("\n" + summary.round(3).to_string(index=False))

    if runs["identical"].all():
        print("\nverified: bit identical weights for", ", ".join(map(str, args.workers)),
              f"workers across {args.repeats} repeats")
    else:
        print(f"\nNOT identical: weights differ by up to {runs['max_abs_diff'].max():.2e} between runs")


    #convergence curve for the poster and val score of trained model
    pd.DataFrame({"step": range(1, len(best_history) + 1), "train_rmse": best_history}).to_csv(
        os.path.join(RESULTS_DIR, "gradient_history.csv"), index=False)
    val_rmse, val_mae = score(best_p, load_arrays(os.path.join(PREP_DIR, "val")))
    print(f"\nGradient descent: val RMSE {val_rmse:.3f} | val MAE {val_mae:.3f}")
    np.savez(os.path.join(RESULTS_DIR, "gradient_weights.npz"), **{k: np.atleast_1d(v) for k, v in best_p.items()})

    if args.baseline:
        #closed form style solver on the same training trips for comparison
        train = pd.read_parquet(prepare_sample(2024, TRAIN_FRAC))
        val = pd.read_parquet(prepare_sample(2025, VAL_FRAC))
        started = perf_counter()
        ridge = make_model("ridge", {"alpha": 1.0}).fit(train[FEATURES], train[TARGET])
        seconds = perf_counter() - started
        pred = ridge.predict(val[FEATURES])
        rmse = float(np.sqrt(np.mean((pred - val[TARGET]) ** 2)))
        mae = float(np.mean(np.abs(pred - val[TARGET])))
        print(f"scikitlearn Ridge: val RMSE {rmse:.3f} | val MAE {mae:.3f} | fit {seconds:.1f} s")

if __name__=="__main__":
    main()

In [ ]:
!{sys.executable} src/gradient_compute.py --workers 1 --baseline

### Sequential vs parallel: gradient computation

200 steps with 1, 2 and 3 workers, 3 repeats each. The final weights must be **bit-identical** for every worker count and repeat.

In [ ]:
!{sys.executable} src/gradient_compute.py --workers 1 2 3 --repeats 3

## Non-Deterministic #2 - THEORY

Floating-point addition is not associative: `(0.1 + 0.2) + 0.3 != (0.3 + 0.2) + 0.1`. In gradient computation, each step adds 8 partial gradient sums. If they're added in the order the workers **finish** (`as_completed`), the order changes from step to step and run to run,so the gradient differs in the last bits and the difference carries through all 200 steps.

This is different from Non-Deterministic #1: there the *order of output rows* changed but here the *numbers themselves* change, even though every input and setting is identical.

**Fix:** combine the shard results in a fixed shard order (`pool.map`), which is what our default `--combine ordered` run above does.

## Non-Deterministic #2 - EVIDENCE

The same training is run 3 times with `--combine completion`.Runs that differ from the first are reported with the largest weight difference;the ordered runs above were all bit-identical.

Because the finish order depends on timing, how many runs differ varies from one execution to the next. In the saved Rockhill run, runs 2 and 3 differed from run 1 by 8.9e-16 and 3.6e-15 (`results/logs/rockhill-20261118-1150/gradient_completion.log`, shown below the cell).

In [ ]:
!{sys.executable} src/gradient_compute.py --workers 3 --combine completion --repeats 3

print("\nSaved Rockhill evidence:")
print(open("results/logs/rockhill-20261118-1150/gradient_completion.log").read())

## Search and gradient: speedup charts

`plot_benchmarks.py` draws the charts from a folder of results: speedup vs workers (with the ideal line),run times,load imbalance and gradient descent convergence.

In [ ]:
%%writefile src/plot_benchmarks.py
# plot_benchmarks.py - charts for the search and gradient benchmarks - Tudor
#
# Reads one or more run folders written by run_benchmarks.sh and saves the charts to results/plots/
#
#   python src/plot_benchmarks.py results/logs/rockhill-20261118-1150
#   python src/plot_benchmarks.py results/logs/<run A> results/logs/<run B>    # compare machines
#   python src/plot_benchmarks.py results --out results/plots/notebook          # after running the notebook

import argparse
import os
import platform

import matplotlib
matplotlib.use("Agg")   # draw straight to files, works on the server too
import matplotlib.pyplot as plt
import pandas as pd

PLOT_DIR = "./results/plots"
TASKS = {"search": "Hyperparameter search", "gradient": "Gradient computation"}

# sized for a printed poster
plt.rcParams.update({
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "xtick.labelsize": 12,
    "ytick.labelsize": 12, "legend.fontsize": 11, "figure.titlesize": 16,
    "axes.spines.top": False, "axes.spines.right": False,
})


def machine_name(run_dir):
    # e.g. "Rockhill (AMD EPYC 9124)" or "Apple M4", from environment.txt
    env_path = os.path.join(run_dir, "environment.txt")
    if not os.path.exists(env_path):   # e.g. results/ written by a notebook run: describe this machine
        return f"{platform.node().split('.')[0]} ({os.cpu_count()} cores)"
    lines = open(env_path).read().splitlines()
    host = next((l.split(":", 1)[1].strip() for l in lines if l.startswith("host:")), "")
    cpu = ""
    for line in lines:
        if line.startswith("Model name:"):
            cpu = line.split(":", 1)[1].replace("16-Core Processor", "").strip()
        elif line.startswith("Apple"):
            cpu = line.strip()
    if host and ":" not in host and cpu:      # a readable host name (Macs often report an address)
        return f"{host.capitalize()} ({cpu})"
    return cpu or host or os.path.basename(run_dir)


def timings(run_dir, task):
    name = "search_timings_grid.csv" if task == "search" else "gradient_timings_ordered.csv"
    return pd.read_csv(os.path.join(run_dir, name))


def source(runs):
    # where the numbers come from, shown under every chart title
    repeats = int(timings(runs[0][1], "search")["runs"].max())
    machines = ", ".join(label for label, _ in runs)
    return f"{machines}  ·  mean of {repeats} runs per point"


def plot_speedup(runs):
    # speedup against worker count for each machine, with the ideal (linear) line
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharey=True)
    for ax, (task, title) in zip(axes, TASKS.items()):
        top = 1
        for i, (label, run_dir) in enumerate(runs):
            t = timings(run_dir, task)
            line, = ax.plot(t["workers"], t["speedup"], marker="o", markersize=8, linewidth=2.5, label=label)
            # label each point (not 1 worker); alternate below/above so two machines don't overlap
            offset = (10, -30) if i % 2 == 0 else (-60, 10)
            for w, s, e in zip(t["workers"], t["speedup"], t["efficiency"]):
                if w > 1:
                    ax.annotate(f"{s:.2f}x ({e:.0%})", (w, s), textcoords="offset points",
                                xytext=offset, fontsize=11, color=line.get_color(), fontweight="bold")
            top = max(top, t["workers"].max())
        ax.plot([1, top], [1, top], linestyle="--", color="grey", label="Ideal (linear)")
        ax.set(title=title, xlabel="Workers", xticks=range(1, top + 1), xlim=(0.8, top + 0.6))
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("Speedup vs 1 worker")
    axes[0].legend(loc="upper left")
    fig.suptitle(f"Speedup: sequential vs parallel\n{source(runs)}  ·  labels: speedup (efficiency)",
                 fontsize=14)
    return fig


SEQ_COLOUR, PAR_COLOUR = "#3a76af", "#ee8636"   # same blue/orange as the preprocessing chart


def plot_times(runs):
    # mean wall-clock time per worker count, error bars = standard deviation over repeats
    # one machine: 1 worker in blue (sequential), more workers in orange (parallel), speedup on top
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
    width = 0.8 / len(runs)
    for ax, (task, title) in zip(axes, TASKS.items()):
        for i, (label, run_dir) in enumerate(runs):
            t = timings(run_dir, task)
            x = t["workers"] + (i - (len(runs) - 1) / 2) * width
            colours = ([SEQ_COLOUR if w == 1 else PAR_COLOUR for w in t["workers"]]
                       if len(runs) == 1 else None)
            bars = ax.bar(x, t["wall_mean"], width, yerr=t["wall_std"].fillna(0), capsize=4,
                          label=label, color=colours)
            ax.bar_label(bars, labels=[f"{v:.1f} s" for v in t["wall_mean"]], label_type="center",
                         color="white", fontweight="bold", fontsize=11)
            if len(runs) == 1:
                ax.bar_label(bars, labels=["" if w == 1 else f"{s:.2f}x"
                                           for w, s in zip(t["workers"], t["speedup"])],
                             padding=6, fontweight="bold", fontsize=12)
        ax.set(title=title, xlabel="Workers", ylabel="Wall-clock time (s)",
               xticks=sorted(timings(runs[0][1], task)["workers"]))
        ax.set_ylim(0, ax.get_ylim()[1] * 1.08)
        ax.grid(axis="y", alpha=0.3)
    if len(runs) > 1:
        axes[0].legend()
    else:
        from matplotlib.patches import Patch
        axes[1].legend(handles=[Patch(color=SEQ_COLOUR, label="Sequential"),
                                Patch(color=PAR_COLOUR, label="Parallel")])
    fig.suptitle(f"Mean run time per worker count\n{source(runs)}  ·  error bars: std over repeats",
                 fontsize=14)
    return fig


def plot_times_compact(runs):
    # short, wide version of plot_times for tight poster space (one machine)
    label, run_dir = runs[0]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
    for ax, (task, title) in zip(axes, TASKS.items()):
        t = timings(run_dir, task)
        colours = [SEQ_COLOUR if w == 1 else PAR_COLOUR for w in t["workers"]]
        bars = ax.bar(t["workers"], t["wall_mean"], 0.7, color=colours)
        ax.bar_label(bars, labels=[f"{v:.0f} s" for v in t["wall_mean"]], label_type="center",
                     color="white", fontweight="bold", fontsize=12)
        ax.bar_label(bars, labels=["" if w == 1 else f"{s:.2f}x"
                                   for w, s in zip(t["workers"], t["speedup"])],
                     padding=4, fontweight="bold", fontsize=13)
        ax.set_ylim(0, t["wall_mean"].max() * 1.22)
        ax.set_yticks([])
        ax.spines["left"].set_visible(False)
        ax.set(xticks=list(t["workers"]), xlabel="Workers")
        ax.set_title(title, fontsize=14)
    from matplotlib.patches import Patch
    axes[1].legend(handles=[Patch(color=SEQ_COLOUR, label="Sequential"),
                            Patch(color=PAR_COLOUR, label="Parallel")], loc="upper right", fontsize=11)
    fig.text(0.5, -0.04, f"{label} · mean wall-clock time of 3 runs · speedup vs 1 worker",
             ha="center", fontsize=11, color="dimgrey")
    return fig


def plot_longest_first(runs):
    # load imbalance: grid order vs longest-first at the same worker count, against the ideal time
    fig, ax = plt.subplots(figsize=(8, 5.2))
    rows = []
    for label, run_dir in runs:
        lf = pd.read_csv(os.path.join(run_dir, "search_timings_longest_first.csv")).iloc[0]
        grid = timings(run_dir, "search")
        g = grid[grid["workers"] == lf["workers"]].iloc[0]
        rows.append((label, g["wall_mean"], lf["wall_mean"], lf["busy_mean"] / lf["workers"], int(lf["workers"])))

    w = 0.27
    x = list(range(len(rows)))
    grid_bars = ax.bar([i - w for i in x], [r[1] for r in rows], w, label="Grid order")
    lf_bars = ax.bar(x, [r[2] for r in rows], w, label="Longest first")
    ideal_bars = ax.bar([i + w for i in x], [r[3] for r in rows], w, color="lightgrey",
                        label="Ideal (total work ÷ workers)")
    ax.bar_label(grid_bars, labels=[f"{r[1]:.1f} s" for r in rows], padding=3, fontsize=11)
    ax.bar_label(lf_bars, labels=[f"{r[2]:.1f} s\n(−{1 - r[2] / r[1]:.0%})" for r in rows],
                 padding=3, fontsize=11, fontweight="bold")
    ax.bar_label(ideal_bars, labels=[f"{r[3]:.1f} s" for r in rows], padding=3, fontsize=11)

    names = [f"{r[4]} workers" if len(rows) == 1 else f"{r[0]}\n({r[4]} workers)" for r in rows]
    ax.set(xticks=x, xticklabels=names, ylabel="Wall-clock time (s)",
           ylim=(0, max(r[1] for r in rows) * 1.45))
    ax.set_title(f"Hyperparameter search: load imbalance\n{source(runs)}", fontsize=14)
    ax.legend(loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    return fig


def plot_convergence(runs):
    # training RMSE per gradient descent step (identical for every worker count)
    label, run_dir = runs[0]
    h = pd.read_csv(os.path.join(run_dir, "gradient_history.csv"))
    t = timings(run_dir, "gradient")

    fig, ax = plt.subplots(figsize=(8, 5.2))
    ax.plot(h["step"], h["train_rmse"], linewidth=2.5)
    first, last = h.iloc[0], h.iloc[-1]
    ax.annotate(f"{first['train_rmse']:.1f} min", (first["step"], first["train_rmse"]),
                textcoords="offset points", xytext=(10, -5), fontsize=11)
    ax.annotate(f"{last['train_rmse']:.2f} min after {int(last['step'])} steps",
                (last["step"], last["train_rmse"]), textcoords="offset points", xytext=(-10, 12),
                ha="right", fontsize=11, fontweight="bold")
    if t["all_identical"].all():
        workers = ", ".join(str(w) for w in t["workers"])
        ax.text(0.98, 0.6, f"Weights bit-identical\nfor {workers} workers",
                transform=ax.transAxes, ha="right", fontsize=11, color="dimgrey")
    ax.set(xlabel="Step", ylabel="Training RMSE (minutes)")
    ax.set_title(f"Gradient descent convergence (data-parallel, shard sums in fixed order)\n{label}",
                 fontsize=14)
    ax.grid(alpha=0.3)
    return fig


def main():
    parser = argparse.ArgumentParser(description="plot benchmark results")
    parser.add_argument("run_dirs", nargs="+", help="run folders (results/logs/<run>, or results/ after a notebook run)")
    parser.add_argument("--out", default=PLOT_DIR, help="folder for the charts")
    args = parser.parse_args()

    runs = [(machine_name(d), d) for d in args.run_dirs]
    os.makedirs(args.out, exist_ok=True)

    for name, make in [("speedup", plot_speedup), ("times", plot_times), ("times_compact", plot_times_compact),
                       ("load_imbalance", plot_longest_first), ("convergence", plot_convergence)]:
        fig = make(runs)
        fig.tight_layout()
        path = os.path.join(args.out, f"{name}.png")
        fig.savefig(path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print("saved", path)


if __name__ == "__main__":
    main()


## Search and gradient: speedup and Amdahl's law

`plot_benchmarks.py` draws the charts from a folder of results: speedup vs workers, run times, load imbalance and gradient descent convergence. The table below summarises the run above on whichever machine executes this notebook; the reported results and charts come from the Rockhill run in the next section.

In [ ]:
import pandas as pd
from IPython.display import Image, display

def amdahl_table(folder):
    # speedup, efficiency and estimated parallel fraction p = (1 - 1/S) / (1 - 1/N) per worker count
    rows = []
    for task, name in [("search", "search_timings_grid.csv"), ("gradient", "gradient_timings_ordered.csv")]:
        t = pd.read_csv(f"{folder}/{name}")
        for r in t[t.workers > 1].itertuples():
            rows.append({"task": task, "workers": r.workers, "time (s)": r.wall_mean,
                         "speedup": r.speedup, "efficiency": r.efficiency,
                         "Amdahl p": (1 - 1 / r.speedup) / (1 - 1 / r.workers)})
    return pd.DataFrame(rows).round(3)

# this run (above), on whichever machine runs the notebook: shows the code runs and scales
# the reported results and charts are the Rockhill run below
display(amdahl_table("results"))

### Rockhill results(used on our poster)

The poster numbers come from one run of the same code on the Rockhill server (2× AMD EPYC 9124, 32 cores, up to 3 workers because the server is shared), saved in `results/logs/rockhill-20261118-1150/` together with the machine details, library versions and full logs. Re-running it needs the server, so the saved results are loaded here.

In [ ]:
ROCKHILL = "results/logs/rockhill-20261118-1150"
!{sys.executable} src/plot_benchmarks.py {ROCKHILL} --out results/plots/rockhill
print(open(f"{ROCKHILL}/environment.txt").read())
display(amdahl_table(ROCKHILL))
for chart in ("times_compact", "load_imbalance"):
    display(Image(f"results/plots/rockhill/{chart}.png", width=900))

### Discussion (Rockhill, 3 workers)

| Task | 1 worker | 3 workers | Speedup | Efficiency | Amdahl p |
|---|---|---|---|---|---|
| Hyperparameter search | 138.5 s | 62.4 s | 2.22× | 74% | around 0.82 |
| Search, longest first | | 48.5 s | 2.86× | 95% | |
| Gradient computation | 100.5 s | 42.4 s | 2.37× | 79% | around 0.8 |

- **Search**: the trials synchronise only once, at the end, so the parallel fraction is high. The main loss is **load imbalance**: three 500-iteration trials take about 30 s each. Longest-first cuts the time by 22% to within 5% of the ideal (total work ÷ workers = 46.1 s); the rest is task granularity (8 uneven tasks cannot be split evenly over 3 workers). Running trials together added only about 2% to the total work, so memory contention is small on this server.
- **Gradient computation**: each step takes 212 ms against an ideal 168 ms, so about 44 ms per step is lost to sending tasks, collecting results and waiting for the slowest shard. That cost is paid at all 200 synchronisation points, which is why this pattern is more sensitive to communication overhead than the search.
- **Amdahl's law**: p is estimated from the measured speedup as p = (1 − 1/S) / (1 − 1/N). It is an *effective* fraction: the truly serial code (combining sums, updating weights) takes well under 1 ms per step; the rest is overhead that behaves like serial work.
- Outputs were verified identical for 1, 2 and 3 workers in every run (validation scores for the search, bit-identical weights for gradient descent).